This is the hardest Spark concept. Almost everyone gets confused because people explain Job, DAG, Stage, and Task separately, when they're actually part of one execution flow.

Let's use a real example.

Imagine you're a restaurant owner.

A customer orders:

🍕 100 pizzas

What happens?

Step 1: The Order (Job)

The customer places one order.

Order:
Deliver 100 pizzas

In Spark, this is a Job.

A job is simply the entire work Spark needs to do after an action like:

df.show()
df.count()
df.write.save()

So:

One action = One Job

Step 2: Planning (DAG)

Before cooking, the manager plans the work.

Receive Order
      │
Prepare Dough
      │
Add Toppings
      │
Bake
      │
Pack
      │
Deliver

This plan is the DAG.

A DAG is not execution.

It's just the execution plan.

Similarly, for Spark:

df.filter(...)
  .groupBy(...)
  .count()

Spark plans:

Read Table
      │
Filter
      │
GroupBy
      │
Count

That's the DAG.

DAG = Spark's blueprint.

Step 3: Breaking the Work (Stages)

Now suppose your restaurant has two kitchens.

Kitchen A prepares the pizzas.

Kitchen B packs and delivers them.

You cannot start packing until cooking finishes.

So the work becomes:

Stage 1

Prepare Dough
Add Toppings
Bake

--------------------

Stage 2

Pack
Deliver

A stage is simply a group of operations that can run together.

In Spark, a new stage usually starts after a Shuffle.

Example
df.filter(...)

No shuffle.

Only one stage.

But

df.groupBy("city").count()

Grouping requires data from different workers.

Spark must exchange data.

That's called Shuffle.

Read

↓

Filter

↓

Shuffle

↓

GroupBy

↓

Count

Spark splits this into:

Stage 1

Read

Filter

Shuffle

-------------------

Stage 2

GroupBy

Count
Step 4: Tasks

Now imagine Stage 1 has to cook 100 pizzas.

Instead of one chef making all 100...

10 chefs each make 10 pizzas.

Chef 1 → 10 pizzas

Chef 2 → 10 pizzas

Chef 3 → 10 pizzas

...

Chef 10 → 10 pizzas

Each chef's work is a Task.

In Spark:

If your data has:

8 partitions

Stage 1 becomes:

Task 1

Task 2

Task 3

...

Task 8

Each task processes one partition.

Put everything together

Suppose you write:

df.filter(df.country == "India") \
  .groupBy("city") \
  .count() \
  .show()

Spark thinks like this.

Action?
show()

Yes.

Create one Job.

Job 1

Now create the DAG.

Read Table
      │
Filter
      │
GroupBy
      │
Count
      │
Show
Shuffle?

Yes.

GroupBy requires shuffle.

So split into stages.

Stage 1

Read

Filter

Shuffle

-------------------

Stage 2

GroupBy

Count

Show
Number of partitions?

Suppose there are 4 partitions.

Stage 1

Task 1

Task 2

Task 3

Task 4

Stage 2

Task 1

Task 2

Task 3

Task 4

Workers execute these tasks.

Final hierarchy
Notebook

↓

Action (show())

↓

Job

↓

DAG (Execution Plan)

↓

Stage 1
    │
    ├── Task 1
    ├── Task 2
    ├── Task 3
    └── Task 4

↓

Stage 2
    │
    ├── Task 1
    ├── Task 2
    ├── Task 3
    └── Task 4

↓

Workers execute tasks

↓

Result returned
One-line definitions
Term	Meaning
Job	The complete work triggered by an action (show, count, write, etc.)
DAG	The execution plan showing the sequence of operations and dependencies
Stage	A chunk of the job that can execute without a shuffle; a shuffle creates a new stage
Task	The smallest unit of work, processing one partition of data within a stage
The relationship is easy to remember:
One Action
      │
      ▼
One Job
      │
      ▼
One DAG (the plan)
      │
      ▼
Multiple Stages
      │
      ▼
Multiple Tasks
      │
      ▼
Worker Nodes execute the Tasks
Here's the key insight that many tutorials skip:
Job answers: "What work needs to be done?"
DAG answers: "In what order should the work happen?"
Stage answers: "Which operations can run together before data must be shuffled?"
Task answers: "How do we divide this stage across the data partitions so workers can execute it in parallel?"

Once you see them as different levels of the same execution process, rather than four unrelated concepts, Spark's execution model becomes much easier to follow.

What you can influence

Although Spark decides how to execute, your code and table design strongly influence its decisions.

For example:

1. Number of partitions
df.repartition(100)

This changes how many tasks Spark creates.

If you have 100 partitions:

Stage 1

Task 1
Task 2
...
Task 100
2. Partitioned tables
CREATE TABLE sales
PARTITIONED BY(country)

Spark can skip entire folders.

3. ZORDER
OPTIMIZE sales
ZORDER BY(customer_id)

Spark reads fewer data blocks.

4. Broadcast joins
from pyspark.sql.functions import broadcast

df1.join(broadcast(df2))

You're giving Spark a hint about a better join strategy.

Think of Spark as Google Maps

You tell Google Maps:

"Take me from Chennai to Bangalore."

You don't tell it:

which road to take,
where to turn,
how to avoid traffic.

Google Maps computes the route.

Spark works similarly.

You tell it:

Read this table.
Filter these rows.
Group by city.
Count them.

Spark decides:

How many stages?
How many tasks?
Which worker runs each task?
Which join algorithm?
Which files to read?
Whether to push filters down to the data source?
In Databricks Free Edition

You configure almost nothing.

Notebook
      │
Managed Serverless Compute
      │
Spark

Databricks manages everything.

In Azure Databricks (Production)

You configure a bit more:

Notebook
      │
Choose Compute
      │
Driver Size
Worker Size
Autoscaling
Runtime
Photon On/Off

Spark still automatically creates:

✅ Jobs
✅ DAGs
✅ Stages
✅ Tasks
Interview answer

If someone asks:

Do we need to manually create Jobs, DAGs, Stages, or Tasks in Spark?

A good answer is:

"No. Spark automatically creates the execution plan, including the DAG, stages, and tasks. As developers, we write transformations and actions, while Spark's Catalyst optimizer and scheduler determine how the work is divided and executed across the cluster. We mainly configure the compute resources and optimize data layout when needed."

That's exactly the abstraction Databricks provides: you focus on the data logic, and Spark handles the distributed execution.

his is another fundamental Spark concept. Once you understand Lazy Evaluation, you'll understand why Spark is so fast.

What is Lazy Evaluation?

Definition:

Spark does not execute transformations immediately. It waits until an action is called, then executes everything together in the most efficient way.

Let's see why.

Imagine you're ordering food

Suppose you tell a waiter:

I want:

🍕 Pizza
🥤 Coke
🍰 Cake

Does the waiter run to the kitchen after every item?

❌ No.

He waits until you've finished ordering.

Then he goes once with the complete order.

Spark behaves exactly the same way.

Your example
df = spark.read.table("employee")

df = df.filter("salary > 50000")

df = df.select("name", "salary")

Let's see what Spark does.

Line 1
df = spark.read.table("employee")

Did Spark read the table?

❌ No.

It only notes:

Later,
I need to read the employee table.
Line 2
df = df.filter("salary > 50000")

Did Spark filter?

❌ No.

It simply adds another step to its plan.

Read employee

↓

Filter salary > 50000
Line 3
df = df.select("name","salary")

Did Spark select the columns?

❌ No.

It extends the plan.

Read employee

↓

Filter salary > 50000

↓

Select name,salary

Still...

No data has been read.

Why?

Because Spark is building the DAG (execution plan).

It wants to know the entire workflow before executing.

When does Spark actually work?

Suppose now you write:

df.show()

Now Spark says:

"Okay, now I have to produce a result."

This is an Action.

Execution begins.

Driver

↓

Create DAG

↓

Optimize DAG

↓

Create Stages

↓

Create Tasks

↓

Workers read Parquet files

↓

Return results
Why is this useful?

Imagine Spark executed every line immediately.

spark.read.table("employee")

Reads 500 GB.

Then

filter(...)

Reads 500 GB again.

Then

select(...)

Reads 500 GB again.

Very slow.

Instead, Spark waits.

It combines everything into one optimized execution.

spark.read.table("employee") \
     .filter("salary>50000") \
     .select("name","salary") \
     .show()

Spark reads the table only once.

Another example

Suppose the table has:

Name	Salary	Age	City	Phone	Department	Manager

You only need:

.select("name","salary")

Since Spark waited, the Catalyst Optimizer says:

"Why read all 7 columns?"

Instead, it reads only:

Name

Salary

This is called Column Pruning.

This optimization is only possible because of Lazy Evaluation.

What if Spark wasn't lazy?
Read Table

↓

Read all columns

↓

Filter

↓

Read again

↓

Select

↓

Read again

Huge waste.

With Lazy Evaluation
Read Table

↓

Filter

↓

Select

↓

Show

Spark builds the complete plan first.

Then executes once.

In [0]:
employees = [
    (1, "Alice", "IT", 90000),
    (2, "Bob", "HR", 60000),
    (3, "Charlie", "Finance", 85000),
    (4, "David", "IT", 75000),
    (5, "Eva", "Marketing", 70000),
    (6, "Frank", "HR", 65000),
    (7, "Grace", "Finance", 95000),
    (8, "Helen", "IT", 80000)
]

emp_df = spark.createDataFrame(
    employees,
    ["emp_id","name","department","salary"]
)

emp_df.write.mode("overwrite").saveAsTable("day7_employee")

In [0]:
departments = [
    ("IT","Technology"),
    ("HR","Human Resources"),
    ("Finance","Finance"),
    ("Marketing","Marketing")
]

dept_df = spark.createDataFrame(
    departments,
    ["department","dept_name"]
)

dept_df.write.mode("overwrite").saveAsTable("day7_department")

In [0]:
emp = spark.table("day7_employee")

dept = spark.table("day7_department")

In [0]:
result = (
    emp
    .filter("salary > 70000")
    .join(dept, "department")
    .groupBy("dept_name")
    .count()
)

In [0]:
result.explain(True)

In [0]:
result.show()

In [0]:
result.count()

In [0]:
result.cache()

In [0]:
result.show()

result.count()

result.collect()

In [0]:
result.explain(True)

In [0]:
%sql
SELECT
    d.dept_name,
    COUNT(*) AS employees
FROM day7_employee e
JOIN day7_department d
ON e.department=d.department
WHERE salary>70000
GROUP BY d.dept_name;

In [0]:
%sql
EXPLAIN SELECT
    d.dept_name,
    COUNT(*) AS employees
FROM day7_employee e
JOIN day7_department d
ON e.department=d.department
WHERE salary>70000
GROUP BY d.dept_name;